In [9]:
import pandas as pd
import pickle
import os
from string import punctuation
import random

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import SnowballStemmer, WordNetLemmatizer
from nltk.tag import pos_tag
from nltk.probability import FreqDist
from nltk.classify import NaiveBayesClassifier, accuracy

### **Variable**

In [ ]:
stemmer = SnowballStemmer('english')
lemma = WordNetLemmatizer()
eng_stopwords = stopwords.words('english')

### **Preprocessing**

In [ ]:
def alter_tag(tag: str):
    if tag.startswith('J'):
        return 'a'
    elif tag.startswith('V'):
        return 'v'
    elif tag.startswith('R'):
        return 'r'
    else:
        return 'n'

def Preprocess(docx):
    tokens = word_tokenize(docx)
    tokens = [tok.lower() for tok in tokens]
    tokens = [tok for tok in tokens if tok.isalpha()]
    tokens = [tok for tok in tokens if tok not in eng_stopwords]
    tokens = [tok for tok in tokens if tok not in punctuation]

    tokens = stemmer.stem(tokens)
    
    tagged = pos_tag(tokens)
    
    tokens = [lemma.lemmatize(tok, alter_tag(tag)) for tok, tag in tagged]

    return tokens


### **Training**

In [11]:
def Training():
    data = pd.read_csv('./Dataset/Tweets.csv')
    X = data['text']
    Y = data['airline_sentiment']

    alldata = ' '.join(X)
    tokens = Preprocess(alldata)
    freq = FreqDist(tokens)

    # Feature Extraction
    feats = []

    for text, label in zip(X, Y):
        clean = Preprocess(text)
        
        feat = {word: True for word in clean}
        feats.append(feat)

    random.shuffle(feats)


    # Training
    split = int(0.8 * len(feats))
    train_data = feats[:split]
    evals_data = feats[split:]

    print('Start Training...')
    model = NaiveBayesClassifier.train(train_data)
    acc = accuracy(model, evals_data)

    print(f'Traning Accuracy: {(acc * 100):2f}')

    # Info
    print('')
    print('Most Informative Features')
    model.show_most_informative_features(5)

    # Save Model
    with open('./model.pickle', 'wb') as file:
        pickle.dump(model, file)
    print('Model Saved')

    return model

def Load():
    if os.path.exists('./model.pickle'):
        file = open('./model.pickle', 'rb')
        model = pickle.load(file)
        file.close()
        return model
    else:
        model = Training()
        return model

### **Support Menu**

In [ ]:
def Analyze_Text(model, text: str):
    

### **Main**

In [ ]:
def Menu():
    text = ''
    model = Load()

    while True:
        print('1. Write Tweet')
        print('2. Analyze Tweet')
        print('3. End Session')

        cc = print('>> ')
        
        if (cc == '1'):
            pass 
        elif (cc == '2'):
            pass
        elif (cc == '3'):
            print('Alright, Thanks for using our App ~~ :)')
            break
        else:
            print('Invalid Input')

In [ ]:
Menu()